In [37]:
# 1. PROCESSAMENTO E CONSOLIDAÇÃO DA BASE

import os
from google.colab import drive
import pandas as pd

drive.mount("/content/drive")

caminho = "/content/drive/MyDrive/AD/data/"

# carregamento
dom1 = pd.read_csv(
    caminho + "ACM_SP.csv", encoding="latin1", sep=";"
).drop(columns=["NM_MUN", "NM_UF", "CD_UF"], errors="ignore")
dom2 = pd.read_csv(
    caminho + "ACM_SP2.csv", encoding="latin1", sep=";"
).drop(columns=["NM_MUN", "NM_UF", "CD_UF"], errors="ignore")
dom3 = pd.read_csv(
    caminho + "ACM_SP3.csv", encoding="latin1", sep=";"
).drop(columns=["NM_MUN", "NM_UF", "CD_UF"], errors="ignore")

basico = pd.read_csv(
    caminho + "ACM_basico_SP.csv",
    encoding="latin1",
    sep=";",
)

df_favelas = pd.read_csv(caminho + "pop_favelas_m.csv",sep=";")

# Limpeza
colunas_remover = [
    "CD_REGIAO",
    "NM_REGIAO",
    "CD_UF",
    "NM_UF",
    "CD_NU",
    "NM_NU",
    "CD_AGLOM",
    "NM_AGLOM",
    "CD_RGINT",
    "NM_RGINT",
    "CD_RGI",
    "NM_RGI",
    "CD_CONCURB",
    "NM_CONCURB",
    "AREA_KM2",
    "v0001",
    "v0002",
    "v0003",
    "v0004",
    "v0006",
]
basico = basico.drop(columns=colunas_remover, errors="ignore")

# Padronização
basico["CD_MUN"] = basico["CD_MUN"].astype(str)
dom1["CD_MUN"] = dom1["CD_MUN"].astype(str)
dom2["CD_MUN"] = dom2["CD_MUN"].astype(str)
dom3["CD_MUN"] = dom3["CD_MUN"].astype(str)
df_favelas["COD_MUNICIPIO"] = df_favelas["COD_MUNICIPIO"].astype(str)

# MERGE
df = (
    basico.merge(dom1, on="CD_MUN", how="left")
    .merge(dom2, on="CD_MUN", how="left")
    .merge(dom3, on="CD_MUN", how="left")
    .merge(df_favelas, left_on="CD_MUN", right_on="COD_MUNICIPIO", how="left")
)

# PÓS-MERGE
# Municípios que não têm favelas vêm como NaN, preenchemos com 0 e convertemos para inteiro
df["N_POP_DPPO"] = df["N_POP_DPPO"].fillna(0).astype(int)

# Remover a coluna de código duplicada que veio da tabela de favelas
df = df.drop(columns=["COD_MUNICIPIO"], errors="ignore")

print(
    f"Base consolidada final: {df.shape[0]} municípios e {df.shape[1]} variáveis."
)

# salvar no drive
# salva usando uma codificação segura para acentos (utf-8-sig)
df.to_csv(caminho + "base_final_SP.csv", index=False, sep=";", encoding="utf-8")
print(
    "gerado com sucesso!"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base consolidada final: 645 municípios e 648 variáveis.
gerado com sucesso!
